**PASS: fresh instructor execution.** All saved outputs below came from this notebook's actual code. Run UTC: `2026-11-04T15:25:41.312301+00:00`; executed 4/4 code cells; 6 output objects. This validates the displayed computations; the described synthetic scope still applies.
<!-- canonical-execution-status:end -->

# 03 · Optional systems lab: RoPE, grouped keys/values, and caching

**Optional follow-up, not required background for the 30-minute core lecture.**
Learn to verify mathematical invariants and compare an optimization against an equivalent baseline.
CPU timing depends on sequence length, interpreter overhead, hardware, and implementation. No speedup is promised.

In [1]:
from pathlib import Path
import sys, math, time, statistics, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'src/vgu_transformers.py').is_file(), 'Run from REHEARSAL_READY or its notebooks directory.'
sys.path.insert(0, str(ROOT / 'src'))
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
import vgu_transformers as vgu
from vgu_transformers import *
torch.manual_seed(9)
torch.set_num_threads(1)
FIGURES=ROOT/'verification/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'figure.facecolor':'white','axes.facecolor':'white'})
print('PyTorch:',torch.__version__,'| device: CPU | threads:',torch.get_num_threads())
print('Actual imported source:',Path(vgu.__file__).resolve().relative_to(ROOT))
assert Path(vgu.__file__).resolve()==(ROOT/'src/vgu_transformers.py').resolve()

PyTorch: 2.10.0+cpu | device: CPU | threads: 1
Actual imported source: src/vgu_transformers.py


## 1. RoPE rotates paired features without changing their squared norm

Position changes the rotation angle applied to queries and keys. It does not rotate values here.
The same frequency convention must be used for Q and K. A cache must start new positions at the correct offset.

In [2]:
x=torch.randn(2,4,9,8)
rotated=rope(x,start_pos=0)
offset=rope(x,start_pos=7)
norm_error=float((rotated.square().sum(-1)-x.square().sum(-1)).abs().max())
torch.testing.assert_close(rotated.square().sum(-1),x.square().sum(-1),atol=2e-6,rtol=2e-6)
print('RoPE max squared-norm difference:',norm_error)
print('rotation differs at shifted positions:',float((rotated-offset).abs().max()))
print('2 new queries after 3 cached tokens use mask:\n',causal_additive_mask(2,t_key=5,past=3))

RoPE max squared-norm difference: 1.9073486328125e-06
rotation differs at shifted positions: 2.270735502243042
2 new queries after 3 cached tokens use mask:
 tensor([[0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])


## 2. Full, token-cached, and chunk-cached outputs must agree

With four query heads, compare one, two, and four KV heads. These correspond to shared, grouped, and per-head K/V.
A cache stores prior K/V; it does not change which past tokens are visible. The full pass is an independent execution path.
Chunk sizes 2, 3, and 4 exercise causal offsets beyond single-token decoding.

In [3]:
torch.manual_seed(19)
for kv_heads in (1,2,4):
    model=GQACausalAttention(d_model=32,n_heads=4,n_kv_heads=kv_heads).eval()
    x=torch.randn(2,9,32)
    with torch.no_grad():
        full=model(x)
        cache=None;token_outputs=[]
        for position in range(9):
            y,cache=model(x[:,position:position+1],past_kv=cache,use_cache=True)
            token_outputs.append(y)
        tokenwise=torch.cat(token_outputs,1)
        chunk_cache=None;chunks=[];start=0
        for length in (2,3,4):
            y,chunk_cache=model(x[:,start:start+length],past_kv=chunk_cache,use_cache=True)
            chunks.append(y);start+=length
        chunkwise=torch.cat(chunks,1)
    torch.testing.assert_close(full,tokenwise,atol=2e-6,rtol=2e-6)
    torch.testing.assert_close(full,chunkwise,atol=2e-6,rtol=2e-6)
    actual_bytes=sum(t.numel()*t.element_size() for t in cache)
    logical_bytes=kv_bytes(2,1,9,kv_heads,8,4)
    assert actual_bytes==logical_bytes
    print('KV heads:',kv_heads,'token error:',float((full-tokenwise).abs().max()),
          'chunk error:',float((full-chunkwise).abs().max()),'KV tensor bytes:',actual_bytes)

KV heads: 1 token error: 7.450580596923828e-08 chunk error: 7.450580596923828e-08 KV tensor bytes: 1152
KV heads: 2 token error: 5.960464477539063e-08 chunk error: 5.960464477539063e-08 KV tensor bytes: 2304
KV heads: 4 token error: 5.960464477539063e-08 chunk error: 5.960464477539063e-08 KV tensor bytes: 4608


## 3. A declared CPU benchmark over three lengths

Compare producing one output per prefix in both paths. The uncached path recomputes the whole growing prefix;
the cached path only computes the new token and retains earlier K/V. Both use the same model and predetermined inputs.
Use one warm-up per path and five timed repetitions per length, with one CPU thread and no gradients.
Report medians in milliseconds. Ratios describe this local toy kernel only and need not exceed one.

In [4]:
benchmark_model=GQACausalAttention(d_model=32,n_heads=4,n_kv_heads=2).eval()
@torch.no_grad()
def cached_decode(x):
    cache=None;parts=[]
    for position in range(x.shape[1]):
        y,cache=benchmark_model(x[:,position:position+1],past_kv=cache,use_cache=True)
        parts.append(y)
    return torch.cat(parts,1)
@torch.no_grad()
def uncached_decode(x):
    return torch.cat([benchmark_model(x[:,:position+1])[:,-1:] for position in range(x.shape[1])],1)
timing_rows=[]
for length in (8,32,64):
    x=torch.randn(1,length,32)
    torch.testing.assert_close(cached_decode(x),uncached_decode(x),atol=2e-6,rtol=2e-6)
    row={'length':length,'warmup_per_path':1,'repetitions':5}
    for name,method in [('cached_ms',cached_decode),('uncached_ms',uncached_decode)]:
        method(x)
        durations=[]
        for _ in range(5):
            start=time.perf_counter();method(x);durations.append(1000*(time.perf_counter()-start))
        row[name]=statistics.median(durations)
    row['uncached_over_cached']=row['uncached_ms']/row['cached_ms']
    timing_rows.append(row)
    print(json.dumps(row))
(ROOT/'verification/cache_benchmark.json').write_text(json.dumps(timing_rows,indent=2)+'\n')
print('Interpret the displayed ratios by length. They are observed local timings, not a universal cache speed claim.')

{"length": 8, "warmup_per_path": 1, "repetitions": 5, "cached_ms": 3.4860060004575644, "uncached_ms": 4.766280999319861, "uncached_over_cached": 1.367261272268106}


{"length": 32, "warmup_per_path": 1, "repetitions": 5, "cached_ms": 17.246220999368234, "uncached_ms": 16.57137199799763, "uncached_over_cached": 0.960869746398627}


{"length": 64, "warmup_per_path": 1, "repetitions": 5, "cached_ms": 27.352594999683788, "uncached_ms": 36.250242999813054, "uncached_over_cached": 1.325294473896613}
Interpret the displayed ratios by length. They are observed local timings, not a universal cache speed claim.


## Reasoning to take forward

Explain why a wrong positional offset can invalidate a cache even when its shapes look correct.
KV tensor bytes here exclude allocator overhead, model weights, activations, and intermediate repeated KV tensors.
The reference GQA implementation repeats KV heads for clarity; it does not establish the performance of a fused serving kernel.
A research extension varies sequence length, batch size, dtype, device, and KV-head count, checks output equivalence first,
then reports timing distributions and total memory with the measurement policy stated.